# 04_figures_tables_and_manuscript_framing.ipynb

Purpose: use the fixed outputs from Notebook 03 to create manuscript-ready tables, figure-data CSV files, Plotly HTML figures, audit files, and manuscript framing text.

This notebook does **not** rerun ML. It expects `outputs.zip` from Notebook 03.


In [ ]:
%pip install -U pandas numpy plotly tabulate openpyxl


## Cell 2 — Imports and folders


In [ ]:
import os
import json
import zipfile
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

BASE_DIR = Path("sodium_cathode_figures_tables_manuscript")
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "outputs"

TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DATA_DIR = OUTPUT_DIR / "figure_data"
HTML_FIG_DIR = OUTPUT_DIR / "html_figures"
MANUSCRIPT_DIR = OUTPUT_DIR / "manuscript_text"
AUDIT_DIR = OUTPUT_DIR / "audit"

for folder in [BASE_DIR, INPUT_DIR, OUTPUT_DIR, TABLE_DIR, FIG_DATA_DIR, HTML_FIG_DIR, MANUSCRIPT_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Notebook 04 project folder:", BASE_DIR.resolve())


## Cell 3 — Locate and extract Notebook 03 outputs


In [ ]:
def find_first_existing(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError("None of these files were found:\\n" + "\\n".join(str(x) for x in candidates))

OUTPUTS_ZIP = find_first_existing([
    Path("outputs.zip"),
    Path("/mnt/data/outputs.zip"),
    Path("sodium_cathode_ml_protocols/outputs.zip"),
])

EXTRACT_DIR = INPUT_DIR / "notebook03_outputs_extracted"

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(OUTPUTS_ZIP, "r") as zf:
    zf.extractall(EXTRACT_DIR)

SOURCE_OUTPUTS_DIR = EXTRACT_DIR / "outputs"
if not SOURCE_OUTPUTS_DIR.exists():
    SOURCE_OUTPUTS_DIR = EXTRACT_DIR

print("Loaded outputs.zip:", OUTPUTS_ZIP)
print("Using extracted outputs directory:", SOURCE_OUTPUTS_DIR)

print("\\nFiles found:")
for p in sorted(SOURCE_OUTPUTS_DIR.rglob("*")):
    if p.is_file():
        print("-", p.relative_to(SOURCE_OUTPUTS_DIR))


## Cell 4 — Load fixed Notebook 03 outputs and validate


In [ ]:
def find_inside(root, filename):
    matches = list(Path(root).rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} inside {root}")
    return matches[0]

PATHS = {
    "master": find_inside(SOURCE_OUTPUTS_DIR, "mp_na_electrodes_master_v2_with_ml_predictions_FIXED.csv"),
    "candidate_all": find_inside(SOURCE_OUTPUTS_DIR, "candidate_shortlist_all_records_FIXED.csv"),
    "candidate_unique": find_inside(SOURCE_OUTPUTS_DIR, "candidate_shortlist_unique_for_manuscript_FIXED.csv"),
    "family_summary": find_inside(SOURCE_OUTPUTS_DIR, "family_level_summary_FIXED.csv"),
    "ml_summary": find_inside(SOURCE_OUTPUTS_DIR, "table_ml_grouped_cv_summary.csv"),
    "ml_best": find_inside(SOURCE_OUTPUTS_DIR, "ml_best_model_by_target_protocol.csv"),
    "feature_importance": find_inside(SOURCE_OUTPUTS_DIR, "extratrees_C_strict_feature_importance.csv"),
    "oof_predictions": find_inside(SOURCE_OUTPUTS_DIR, "ml_grouped_cv_oof_predictions_all_targets.csv"),
    "feature_manifest": find_inside(SOURCE_OUTPUTS_DIR, "protocol_feature_manifest.json"),
}

df_master = pd.read_csv(PATHS["master"])
df_candidate_all = pd.read_csv(PATHS["candidate_all"])
df_candidate_unique = pd.read_csv(PATHS["candidate_unique"])
df_family = pd.read_csv(PATHS["family_summary"])
df_ml_summary = pd.read_csv(PATHS["ml_summary"])
df_ml_best = pd.read_csv(PATHS["ml_best"])
df_importance = pd.read_csv(PATHS["feature_importance"])
df_oof = pd.read_csv(PATHS["oof_predictions"])

with open(PATHS["feature_manifest"], "r", encoding="utf-8") as f:
    feature_manifest = json.load(f)

print("Loaded data shapes:")
for name, obj in [
    ("Master", df_master), ("Candidate all", df_candidate_all), ("Candidate unique", df_candidate_unique),
    ("Family summary", df_family), ("ML summary", df_ml_summary), ("ML best", df_ml_best),
    ("Feature importance", df_importance), ("OOF predictions", df_oof),
]:
    print(name + ":", obj.shape)

assert len(df_master) == 416, f"Expected 416 master rows, got {len(df_master)}"
assert df_master["mp_index"].nunique() == 416, "mp_index is not unique in master."
assert df_master["mp_index"].duplicated().sum() == 0, "Duplicate mp_index found."
assert len(df_candidate_all) <= len(df_master), "Candidate-all table is inflated."
assert len(df_candidate_unique) <= len(df_candidate_all), "Unique candidate table is larger than all candidates."
assert int(df_family["n_records"].sum()) == len(df_master), "Family summary count does not equal master row count."

df_input_audit = pd.DataFrame([
    {"check": "master_rows", "value": len(df_master)},
    {"check": "unique_mp_index", "value": df_master["mp_index"].nunique()},
    {"check": "duplicate_mp_index", "value": int(df_master["mp_index"].duplicated().sum())},
    {"check": "candidate_all_rows", "value": len(df_candidate_all)},
    {"check": "candidate_unique_rows", "value": len(df_candidate_unique)},
    {"check": "family_summary_total_records", "value": int(df_family["n_records"].sum())},
    {"check": "ml_summary_rows", "value": len(df_ml_summary)},
    {"check": "protocol_A_feature_count", "value": len(feature_manifest.get("Protocol_A_composition_only", []))},
    {"check": "protocol_B_feature_count", "value": len(feature_manifest.get("Protocol_B_composition_plus_lattice", []))},
    {"check": "protocol_C_feature_count", "value": len(feature_manifest.get("Protocol_C_strict_post_DFT", []))},
])
audit_path = AUDIT_DIR / "notebook04_input_audit.csv"
df_input_audit.to_csv(audit_path, index=False)

print("\\nAll input checks passed.")
print("Saved:", audit_path)
display(df_input_audit)


## Cell 5 — Helper functions and cleanup


In [ ]:
def clean_bool_series(s):
    return (
        s.astype(str).str.strip().str.lower()
        .map({"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False})
        .fillna(False)
    )

def clean_numeric(df, cols):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def save_table(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    print("Saved:", path, "| shape:", df.shape)

def round_existing(df, cols, ndigits=3):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").round(ndigits)
    return df

bool_cols = [
    "basic_screening_pass", "hard_exclusion_free_candidate", "conservative_earth_abundant_candidate",
    "cde_exact_formula_match", "cde_has_capacity_evidence", "cde_has_voltage_evidence",
    "cde_has_capacity_and_voltage_evidence",
]
for current_df in [df_master, df_candidate_all, df_candidate_unique]:
    for col in bool_cols:
        if col in current_df.columns:
            current_df[col] = clean_bool_series(current_df[col])

numeric_cols = [
    "average_voltage", "capacity_grav", "energy_grav", "max_delta_volume", "stability_charge",
    "stability_discharge", "stability_worst", "mp_preliminary_score", "final_decision_support_score",
    "literature_support_score", "cde_capacity_median", "cde_voltage_median", "cde_energy_median",
    "cde_total_doi_count",
]
df_master = clean_numeric(df_master, numeric_cols)
df_candidate_all = clean_numeric(df_candidate_all, numeric_cols)
df_candidate_unique = clean_numeric(df_candidate_unique, numeric_cols)

print("Cleanup complete.")


## Cell 6 — Dataset funnel table and figure


In [ ]:
total_records = len(df_master)
basic_pass = int(df_master["basic_screening_pass"].sum())
hard_free = int(df_master["hard_exclusion_free_candidate"].sum())
conservative = int(df_master["conservative_earth_abundant_candidate"].sum())
exact_cde = int(df_master["cde_exact_formula_match"].sum())
conservative_cde = int((df_master["conservative_earth_abundant_candidate"] & df_master["cde_exact_formula_match"]).sum())

df_funnel = pd.DataFrame([
    {"stage_order": 1, "stage": "Materials Project Na-ion electrode records", "count": total_records, "interpretation": "Full computational Na-electrode dataset used for screening and ML."},
    {"stage_order": 2, "stage": "Basic electrochemical screening pass", "count": basic_pass, "interpretation": "Records satisfying basic electrochemical criteria."},
    {"stage_order": 3, "stage": "Hard-exclusion-free candidates", "count": hard_free, "interpretation": "Candidates avoiding hard-excluded elements."},
    {"stage_order": 4, "stage": "Conservative earth-abundant candidates", "count": conservative, "interpretation": "Most conservative candidate subset after excluding hard and soft penalty elements."},
    {"stage_order": 5, "stage": "Conservative candidates with exact CDE evidence", "count": conservative_cde, "interpretation": "Conservative candidates with exact formula-level text-mined literature evidence."},
    {"stage_order": 6, "stage": "All final candidate shortlist records", "count": len(df_candidate_all), "interpretation": "Basic screening pass and hard-exclusion-free candidate table."},
    {"stage_order": 7, "stage": "Unique manuscript shortlist", "count": len(df_candidate_unique), "interpretation": "Deduplicated candidate list for manuscript tables."},
])
df_funnel["percentage_of_total"] = (100 * df_funnel["count"] / total_records).round(2)

save_table(df_funnel, TABLE_DIR / "table_1_dataset_funnel.csv")
save_table(df_funnel[["stage_order", "stage", "count", "percentage_of_total"]], FIG_DATA_DIR / "figure_data_candidate_funnel.csv")

fig_funnel = px.funnel(
    df_funnel.sort_values("stage_order"),
    x="count", y="stage", title="Candidate screening funnel",
    hover_data=["percentage_of_total", "interpretation"],
)
fig_funnel.update_layout(width=900, height=550, yaxis_title="", xaxis_title="Number of records")
funnel_html_path = HTML_FIG_DIR / "figure_2_candidate_screening_funnel.html"
fig_funnel.write_html(funnel_html_path)
print("Saved:", funnel_html_path)
display(df_funnel)
fig_funnel.show()


## Cell 7 — ML performance table and figures


In [ ]:
df_ml_table = df_ml_summary.copy()
df_ml_table = df_ml_table.rename(columns={
    "target": "Target property", "protocol": "Protocol", "model": "Best model",
    "n_samples": "Samples", "n_features": "Features", "n_groups": "CV groups",
    "rmse": "RMSE", "mae": "MAE", "r2": "R2", "spearman": "Spearman",
    "top20_enrichment": "Top-20% enrichment", "baseline_rmse": "Baseline RMSE",
    "baseline_mae": "Baseline MAE",
})
df_ml_table = round_existing(df_ml_table, ["RMSE", "MAE", "R2", "Spearman", "Top-20% enrichment", "Baseline RMSE", "Baseline MAE"], 4)
save_table(df_ml_table, TABLE_DIR / "table_2_ml_grouped_cv_summary_manuscript.csv")

df_ml_fig = df_ml_summary.copy()
df_ml_fig["protocol_label"] = df_ml_fig["protocol"].replace({
    "A_composition_only": "A: composition",
    "B_composition_lattice": "B: composition + lattice",
    "C_strict_post_DFT": "C-strict: post-DFT",
})
save_table(df_ml_fig, FIG_DATA_DIR / "figure_data_ml_protocol_comparison.csv")

fig_r2 = px.bar(
    df_ml_fig, x="target", y="r2", color="protocol_label", barmode="group",
    hover_data=["model", "rmse", "mae", "spearman", "top20_enrichment"],
    title="Grouped-CV model performance by protocol: R²",
)
fig_r2.update_layout(width=950, height=520, xaxis_title="Target property", yaxis_title="R²", legend_title="Protocol")
r2_html_path = HTML_FIG_DIR / "figure_4a_ml_protocol_r2_comparison.html"
fig_r2.write_html(r2_html_path)
print("Saved:", r2_html_path)

fig_rmse = px.bar(
    df_ml_fig, x="target", y="rmse", color="protocol_label", barmode="group",
    hover_data=["model", "mae", "r2", "spearman"],
    title="Grouped-CV model performance by protocol: RMSE",
)
fig_rmse.update_layout(width=950, height=520, xaxis_title="Target property", yaxis_title="RMSE", legend_title="Protocol")
rmse_html_path = HTML_FIG_DIR / "figure_4b_ml_protocol_rmse_comparison.html"
fig_rmse.write_html(rmse_html_path)
print("Saved:", rmse_html_path)

display(df_ml_table)
fig_r2.show()
fig_rmse.show()


## Cell 8 — Family-level summary table and figures


In [ ]:
df_family_table = df_family.copy()
df_family_table["basic_screening_rate_percent"] = (100 * df_family_table["n_basic_screening"] / df_family_table["n_records"]).round(2)
df_family_table["hard_exclusion_free_rate_percent"] = (100 * df_family_table["n_hard_exclusion_free"] / df_family_table["n_records"]).round(2)
df_family_table["conservative_rate_percent"] = (100 * df_family_table["n_conservative"] / df_family_table["n_records"]).round(2)
df_family_table["exact_cde_rate_percent"] = (100 * df_family_table["n_exact_cde"] / df_family_table["n_records"]).round(2)
df_family_table = round_existing(df_family_table, ["median_voltage", "median_capacity", "median_energy", "median_mp_score"], 3)

save_table(df_family_table, TABLE_DIR / "table_3_family_level_summary_manuscript.csv")
save_table(df_family_table, FIG_DATA_DIR / "figure_data_family_level_summary.csv")

df_family_long = df_family_table.melt(
    id_vars=["preliminary_family"],
    value_vars=["n_records", "n_basic_screening", "n_hard_exclusion_free", "n_conservative", "n_exact_cde"],
    var_name="category", value_name="count",
)
df_family_long["category"] = df_family_long["category"].replace({
    "n_records": "All records", "n_basic_screening": "Basic pass",
    "n_hard_exclusion_free": "Hard-exclusion-free", "n_conservative": "Conservative",
    "n_exact_cde": "Exact CDE evidence",
})
fig_family_counts = px.bar(df_family_long, x="preliminary_family", y="count", color="category", barmode="group", title="Family-level screening and evidence summary")
fig_family_counts.update_layout(width=1100, height=600, xaxis_title="Candidate family", yaxis_title="Number of records", legend_title="Category", xaxis_tickangle=-25)
family_counts_html_path = HTML_FIG_DIR / "figure_3a_family_level_counts.html"
fig_family_counts.write_html(family_counts_html_path)
print("Saved:", family_counts_html_path)

fig_family_energy = px.scatter(
    df_family_table, x="median_voltage", y="median_energy", size="n_conservative", color="preliminary_family",
    hover_data=["n_records", "n_basic_screening", "n_hard_exclusion_free", "n_conservative", "n_exact_cde", "median_capacity"],
    title="Family-level voltage-energy landscape",
)
fig_family_energy.update_layout(width=850, height=600, xaxis_title="Median average voltage (V)", yaxis_title="Median gravimetric energy (Wh/kg)", legend_title="Family")
family_energy_html_path = HTML_FIG_DIR / "figure_3b_family_voltage_energy_landscape.html"
fig_family_energy.write_html(family_energy_html_path)
print("Saved:", family_energy_html_path)

display(df_family_table)
fig_family_counts.show()
fig_family_energy.show()


## Cell 9 — Candidate shortlist tables and maps


In [ ]:
candidate_cols = [
    "mp_index", "battery_formula", "formula_discharge", "framework_formula", "preliminary_family",
    "average_voltage", "capacity_grav", "energy_grav", "max_delta_volume",
    "stability_charge", "stability_discharge", "mp_preliminary_score",
    "final_decision_support_score", "conservative_earth_abundant_candidate",
    "cde_exact_formula_match", "cde_total_doi_count", "cde_capacity_median",
    "cde_voltage_median", "cde_energy_median",
]
candidate_cols = [c for c in candidate_cols if c in df_candidate_unique.columns]

df_top20 = df_candidate_unique.sort_values(
    by=["conservative_earth_abundant_candidate", "cde_exact_formula_match", "final_decision_support_score"],
    ascending=[False, False, False],
)[candidate_cols].head(20).copy()
df_top20.insert(0, "rank", range(1, len(df_top20) + 1))
df_top20 = round_existing(df_top20, [
    "average_voltage", "capacity_grav", "energy_grav", "max_delta_volume",
    "stability_charge", "stability_discharge", "mp_preliminary_score",
    "final_decision_support_score", "cde_capacity_median", "cde_voltage_median", "cde_energy_median",
], 3)
save_table(df_top20, TABLE_DIR / "table_4_top20_candidate_shortlist_manuscript.csv")

df_top_conservative_cde = df_candidate_unique[
    (df_candidate_unique["conservative_earth_abundant_candidate"] == True)
    & (df_candidate_unique["cde_exact_formula_match"] == True)
].sort_values(by="final_decision_support_score", ascending=False).copy()
save_table(round_existing(df_top_conservative_cde[candidate_cols], [
    "average_voltage", "capacity_grav", "energy_grav", "mp_preliminary_score", "final_decision_support_score",
], 3), TABLE_DIR / "table_top_conservative_candidates_with_cde_evidence.csv")

df_candidate_map = df_candidate_unique.copy()
df_candidate_map["evidence_class"] = np.where(
    (df_candidate_map["conservative_earth_abundant_candidate"] == True) & (df_candidate_map["cde_exact_formula_match"] == True),
    "Conservative + exact CDE evidence",
    np.where(df_candidate_map["conservative_earth_abundant_candidate"] == True,
             "Conservative, computational support only",
             np.where(df_candidate_map["cde_exact_formula_match"] == True, "Non-conservative + exact CDE evidence", "Computational support only"))
)
map_cols = [c for c in [
    "mp_index", "battery_formula", "formula_discharge", "framework_formula", "preliminary_family",
    "average_voltage", "capacity_grav", "energy_grav", "max_delta_volume", "stability_charge",
    "stability_discharge", "mp_preliminary_score", "final_decision_support_score",
    "conservative_earth_abundant_candidate", "cde_exact_formula_match", "cde_total_doi_count", "evidence_class",
] if c in df_candidate_map.columns]
save_table(df_candidate_map[map_cols], FIG_DATA_DIR / "figure_data_candidate_voltage_capacity_energy_map.csv")

fig_candidate_map = px.scatter(
    df_candidate_map, x="average_voltage", y="capacity_grav", size="energy_grav",
    color="preliminary_family", symbol="evidence_class", hover_name="battery_formula",
    hover_data=["formula_discharge", "framework_formula", "energy_grav", "max_delta_volume", "stability_charge", "stability_discharge", "final_decision_support_score", "cde_total_doi_count"],
    title="Candidate map: voltage vs capacity with energy as marker size",
)
fig_candidate_map.update_layout(width=1050, height=700, xaxis_title="Average voltage (V)", yaxis_title="Gravimetric capacity (mAh/g)", legend_title="Family / evidence")
candidate_map_html_path = HTML_FIG_DIR / "figure_5_candidate_voltage_capacity_energy_map.html"
fig_candidate_map.write_html(candidate_map_html_path)
print("Saved:", candidate_map_html_path)

fig_energy_voltage = px.scatter(
    df_candidate_map, x="average_voltage", y="energy_grav", color="preliminary_family", symbol="evidence_class",
    hover_name="battery_formula", hover_data=["formula_discharge", "framework_formula", "capacity_grav", "final_decision_support_score", "cde_total_doi_count"],
    title="Candidate map: voltage vs gravimetric energy",
)
fig_energy_voltage.update_layout(width=1050, height=700, xaxis_title="Average voltage (V)", yaxis_title="Gravimetric energy (Wh/kg)", legend_title="Family / evidence")
energy_voltage_html_path = HTML_FIG_DIR / "figure_5b_candidate_voltage_energy_map.html"
fig_energy_voltage.write_html(energy_voltage_html_path)
print("Saved:", energy_voltage_html_path)

display(df_top20)
fig_candidate_map.show()
fig_energy_voltage.show()


## Cell 10 — Feature importance


In [ ]:
df_importance_clean = df_importance.copy()
df_importance_clean["importance"] = pd.to_numeric(df_importance_clean["importance"], errors="coerce")

def clean_feature_name(x):
    x = str(x)
    x = x.replace("comp__", "")
    x = x.replace("host_structure__lattice__", "lattice_")
    x = x.replace("_", " ")
    return x

df_importance_clean["feature_clean"] = df_importance_clean["feature"].map(clean_feature_name)

df_top_importance = (
    df_importance_clean
    .sort_values(["target", "importance"], ascending=[True, False])
    .groupby("target")
    .head(20)
    .copy()
)
df_top_importance["importance"] = df_top_importance["importance"].round(5)

save_table(df_top_importance, TABLE_DIR / "table_feature_importance_top20_by_target.csv")
save_table(df_top_importance, FIG_DATA_DIR / "figure_data_feature_importance_top20_by_target.csv")

for target_name in sorted(df_top_importance["target"].unique()):
    tmp = df_top_importance[df_top_importance["target"] == target_name].sort_values("importance", ascending=True).copy()
    fig = px.bar(tmp, x="importance", y="feature_clean", orientation="h", title=f"Top C-strict feature importances: {target_name}", hover_data=["feature"])
    fig.update_layout(width=900, height=650, xaxis_title="ExtraTrees feature importance", yaxis_title="Feature")
    out_path = HTML_FIG_DIR / f"figure_6_feature_importance_{target_name}.html"
    fig.write_html(out_path)
    print("Saved:", out_path)
    fig.show()

display(df_top_importance.head(40))


## Cell 11 — Parity plot data and HTML figures


In [ ]:
df_best_c = df_ml_best[df_ml_best["protocol"] == "C_strict_post_DFT"].copy()
parity_rows = []

for _, row in df_best_c.iterrows():
    target_name = row["target"]
    best_model = row["model"]

    tmp = df_oof[
        (df_oof["target"] == target_name)
        & (df_oof["protocol"] == "C_strict_post_DFT")
        & (df_oof["model"] == best_model)
    ].copy()

    if tmp.empty:
        continue

    tmp["target_property"] = target_name
    tmp["best_model"] = best_model
    parity_rows.append(tmp)

    parity_data_path = FIG_DATA_DIR / f"figure_data_parity_{target_name}_C_strict_{best_model}.csv"
    save_table(tmp, parity_data_path)

    min_val = float(np.nanmin([tmp["y_true"].min(), tmp["y_pred_oof"].min()]))
    max_val = float(np.nanmax([tmp["y_true"].max(), tmp["y_pred_oof"].max()]))

    fig = px.scatter(
        tmp, x="y_true", y="y_pred_oof", color="preliminary_family", hover_name="battery_formula",
        hover_data=["formula_discharge", "framework_formula", "abs_error"],
        title=f"Grouped-CV parity plot: {target_name} | C-strict | {best_model}",
    )
    fig.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val], mode="lines", name="Ideal prediction"))
    fig.update_layout(width=800, height=700, xaxis_title=f"Computed {target_name}", yaxis_title=f"Grouped-CV predicted {target_name}")
    out_html = HTML_FIG_DIR / f"figure_parity_{target_name}_C_strict_{best_model}.html"
    fig.write_html(out_html)
    print("Saved:", out_html)
    fig.show()

if parity_rows:
    df_parity_all = pd.concat(parity_rows, ignore_index=True)
    save_table(df_parity_all, FIG_DATA_DIR / "figure_data_parity_all_best_C_strict_models.csv")


## Cell 12 — Supplementary manifest and manuscript framing


In [ ]:
manifest_rows = []

def add_manifest(path, description, recommended_location):
    path = Path(path)
    if path.exists():
        rows, cols = np.nan, np.nan
        if path.suffix.lower() == ".csv":
            try:
                tmp = pd.read_csv(path)
                rows, cols = tmp.shape
            except Exception:
                pass
        manifest_rows.append({
            "file": str(path.relative_to(OUTPUT_DIR)) if OUTPUT_DIR in path.parents else str(path),
            "exists": True, "rows": rows, "columns": cols,
            "description": description, "recommended_location": recommended_location,
        })
    else:
        manifest_rows.append({
            "file": str(path), "exists": False, "rows": np.nan, "columns": np.nan,
            "description": description, "recommended_location": recommended_location,
        })

add_manifest(TABLE_DIR / "table_1_dataset_funnel.csv", "Dataset and candidate filtering funnel.", "Main Table 1 or Figure 2 data")
add_manifest(TABLE_DIR / "table_2_ml_grouped_cv_summary_manuscript.csv", "Grouped-CV ML performance summary.", "Main Table 2")
add_manifest(TABLE_DIR / "table_3_family_level_summary_manuscript.csv", "Family-level candidate and evidence summary.", "Main Table 3")
add_manifest(TABLE_DIR / "table_4_top20_candidate_shortlist_manuscript.csv", "Top 20 ranked candidate shortlist.", "Main Table 4")
add_manifest(TABLE_DIR / "table_feature_importance_top20_by_target.csv", "Top feature importances by target property.", "Supplementary Table")
add_manifest(FIG_DATA_DIR / "figure_data_candidate_voltage_capacity_energy_map.csv", "Candidate map plotting data.", "Figure data / Supplementary")
add_manifest(FIG_DATA_DIR / "figure_data_ml_protocol_comparison.csv", "ML protocol comparison plotting data.", "Figure data / Supplementary")
add_manifest(FIG_DATA_DIR / "figure_data_family_level_summary.csv", "Family-level plotting data.", "Figure data / Supplementary")
add_manifest(FIG_DATA_DIR / "figure_data_feature_importance_top20_by_target.csv", "Feature importance plotting data.", "Figure data / Supplementary")

df_file_manifest = pd.DataFrame(manifest_rows)
manifest_path = AUDIT_DIR / "supplementary_file_manifest.csv"
save_table(df_file_manifest, manifest_path)

n_total = len(df_master)
n_basic = int(df_master["basic_screening_pass"].sum())
n_hard = int(df_master["hard_exclusion_free_candidate"].sum())
n_cons = int(df_master["conservative_earth_abundant_candidate"].sum())
n_cde = int(df_master["cde_exact_formula_match"].sum())
n_cons_cde = int((df_master["conservative_earth_abundant_candidate"] & df_master["cde_exact_formula_match"]).sum())
n_unique_shortlist = len(df_candidate_unique)
n_all_shortlist = len(df_candidate_all)

best_c = df_ml_best[df_ml_best["protocol"] == "C_strict_post_DFT"].copy()
def get_metric(target, metric):
    row = best_c[best_c["target"] == target]
    if row.empty:
        return np.nan
    return row.iloc[0][metric]

top_families = df_family.sort_values("n_conservative", ascending=False).head(3)
top_family_text = "; ".join([f"{r['preliminary_family']} ({int(r['n_conservative'])} conservative candidates)" for _, r in top_families.iterrows()])

methods_text = f"""
# Methods framing text

This study used a reproducible materials-informatics workflow for sodium-ion cathode screening. The workflow began from {n_total} Materials Project sodium-ion electrode records. Candidate filtering was performed using electrochemical screening metrics, hard-exclusion sustainability rules, and a conservative earth-abundance label. Text-mined ChemDataExtractor records were used only as external literature-evidence indicators, not as primary machine-learning targets or main ML input features.

Three ML descriptor protocols were evaluated. Protocol A used composition-derived descriptors only. Protocol B extended Protocol A with available structural or lattice descriptors. Protocol C-strict added post-DFT electrode descriptors and is therefore interpreted as a post-computation decision-support model rather than a pre-DFT discovery model. Model performance was evaluated using grouped cross-validation with framework-level grouping to reduce leakage from related electrode entries.
"""

results_text = f"""
# Results framing text

The initial Materials Project sodium-electrode dataset contained {n_total} records. Of these, {n_basic} passed the basic electrochemical screening criteria, {n_hard} were hard-exclusion-free candidates, and {n_cons} satisfied the conservative earth-abundant candidate definition. Exact formula-level text-mined literature evidence was found for {n_cde} Materials Project electrode records, including {n_cons_cde} conservative earth-abundant candidates. After deduplication, the manuscript-level shortlist contained {n_unique_shortlist} unique candidate entries from {n_all_shortlist} hard-exclusion-free candidate records.

Family-level analysis showed that the strongest conservative candidate representation came from: {top_family_text}. Phosphate or NASICON-like materials gave the broadest balance of screening success and conservative composition, while transition-metal oxide-like materials showed the strongest exact text-mined literature support.

Grouped-CV ML results showed useful predictive performance for voltage, capacity, and gravimetric energy. In the C-strict protocol, the average-voltage model achieved R2 = {get_metric('average_voltage', 'r2'):.3f} and Spearman = {get_metric('average_voltage', 'spearman'):.3f}; the capacity model achieved R2 = {get_metric('capacity_grav', 'r2'):.3f} and Spearman = {get_metric('capacity_grav', 'spearman'):.3f}; and the energy model achieved R2 = {get_metric('energy_grav', 'r2'):.3f} and Spearman = {get_metric('energy_grav', 'spearman'):.3f}. Stability prediction was weaker, with R2 = {get_metric('stability_worst', 'r2'):.3f} and Spearman = {get_metric('stability_worst', 'spearman'):.3f}, and should therefore be interpreted cautiously.
"""

limitations_text = """
# Claim-control and limitations text

The candidate shortlist should be interpreted as a decision-support ranking, not as experimental validation. Materials Project values are computed descriptors and may depend on database settings, calculation choices, and electrode construction assumptions. ChemDataExtractor evidence is text-mined literature evidence and should not be treated as clean experimental ground truth. Exact formula matching can miss relevant literature where formulas are written in non-standard forms, abbreviations, doped compositions, carbon composites, or hydrated/phase-specific notation.

Protocol C-strict includes post-DFT electrode descriptors; therefore, it is not a pre-DFT discovery model. Its value is in ranking, interpretation, and consistency checking after computed electrode data are available. Stability prediction showed weaker grouped-CV performance than voltage, capacity, and energy prediction, so stability-related claims should remain conservative.
"""

manuscript_text_path = MANUSCRIPT_DIR / "manuscript_framing_methods_results_limitations.md"
with open(manuscript_text_path, "w", encoding="utf-8") as f:
    f.write(methods_text + "\\n\\n" + results_text + "\\n\\n" + limitations_text)
print("Saved:", manuscript_text_path)

display(df_file_manifest)
print(methods_text)
print(results_text)
print(limitations_text)


## Cell 13 — Figure/table checklist and final output audit


In [ ]:
checklist_rows = [
    {"item": "Figure 1", "recommended_content": "Workflow diagram: MP extraction → correction → screening → CDE evidence → ML → shortlist.", "created_in_notebook04": "No, create manually or in next graphics notebook.", "source_file": "Use full project workflow."},
    {"item": "Figure 2", "recommended_content": "Candidate screening funnel.", "created_in_notebook04": "Yes", "source_file": "figure_data/figure_data_candidate_funnel.csv and html_figures/figure_2_candidate_screening_funnel.html"},
    {"item": "Figure 3", "recommended_content": "Family-level screening and evidence comparison.", "created_in_notebook04": "Yes", "source_file": "figure_data/figure_data_family_level_summary.csv and html_figures/figure_3a_family_level_counts.html"},
    {"item": "Figure 4", "recommended_content": "ML performance by protocol and target.", "created_in_notebook04": "Yes", "source_file": "figure_data/figure_data_ml_protocol_comparison.csv and html_figures/figure_4a_ml_protocol_r2_comparison.html"},
    {"item": "Figure 5", "recommended_content": "Candidate voltage-capacity-energy map.", "created_in_notebook04": "Yes", "source_file": "figure_data/figure_data_candidate_voltage_capacity_energy_map.csv and html_figures/figure_5_candidate_voltage_capacity_energy_map.html"},
    {"item": "Figure 6", "recommended_content": "Feature importance summary.", "created_in_notebook04": "Yes", "source_file": "figure_data/figure_data_feature_importance_top20_by_target.csv and html_figures/figure_6_feature_importance_*.html"},
    {"item": "Table 1", "recommended_content": "Dataset and filtering funnel.", "created_in_notebook04": "Yes", "source_file": "tables/table_1_dataset_funnel.csv"},
    {"item": "Table 2", "recommended_content": "Grouped-CV ML performance.", "created_in_notebook04": "Yes", "source_file": "tables/table_2_ml_grouped_cv_summary_manuscript.csv"},
    {"item": "Table 3", "recommended_content": "Family-level summary.", "created_in_notebook04": "Yes", "source_file": "tables/table_3_family_level_summary_manuscript.csv"},
    {"item": "Table 4", "recommended_content": "Top 20 candidate shortlist.", "created_in_notebook04": "Yes", "source_file": "tables/table_4_top20_candidate_shortlist_manuscript.csv"},
]
df_checklist = pd.DataFrame(checklist_rows)
checklist_path = AUDIT_DIR / "figure_table_checklist.csv"
save_table(df_checklist, checklist_path)

required_outputs = [
    TABLE_DIR / "table_1_dataset_funnel.csv",
    TABLE_DIR / "table_2_ml_grouped_cv_summary_manuscript.csv",
    TABLE_DIR / "table_3_family_level_summary_manuscript.csv",
    TABLE_DIR / "table_4_top20_candidate_shortlist_manuscript.csv",
    TABLE_DIR / "table_feature_importance_top20_by_target.csv",
    FIG_DATA_DIR / "figure_data_candidate_funnel.csv",
    FIG_DATA_DIR / "figure_data_ml_protocol_comparison.csv",
    FIG_DATA_DIR / "figure_data_family_level_summary.csv",
    FIG_DATA_DIR / "figure_data_candidate_voltage_capacity_energy_map.csv",
    FIG_DATA_DIR / "figure_data_feature_importance_top20_by_target.csv",
    MANUSCRIPT_DIR / "manuscript_framing_methods_results_limitations.md",
    AUDIT_DIR / "notebook04_input_audit.csv",
    AUDIT_DIR / "supplementary_file_manifest.csv",
    AUDIT_DIR / "figure_table_checklist.csv",
]

audit_final_rows = []
for path in required_outputs:
    path = Path(path)
    exists = path.exists()
    size_kb = path.stat().st_size / 1024 if exists else 0
    audit_final_rows.append({"file": str(path.relative_to(OUTPUT_DIR)), "exists": exists, "size_kb": round(size_kb, 2)})

df_final_audit = pd.DataFrame(audit_final_rows)
final_audit_path = AUDIT_DIR / "notebook04_final_output_audit.csv"
df_final_audit.to_csv(final_audit_path, index=False)
print("Saved:", final_audit_path)
display(df_checklist)
display(df_final_audit)

missing = df_final_audit[df_final_audit["exists"] == False]
if len(missing) == 0:
    print("\\nAll required Notebook 04 outputs were generated successfully.")
else:
    display(missing)
    raise FileNotFoundError("Some required Notebook 04 outputs are missing.")


## Files to upload after running Notebook 04

Upload either the complete folder:

`sodium_cathode_figures_tables_manuscript/outputs`

or these key files:
- `tables/table_1_dataset_funnel.csv`
- `tables/table_2_ml_grouped_cv_summary_manuscript.csv`
- `tables/table_3_family_level_summary_manuscript.csv`
- `tables/table_4_top20_candidate_shortlist_manuscript.csv`
- `tables/table_feature_importance_top20_by_target.csv`
- `figure_data/figure_data_candidate_voltage_capacity_energy_map.csv`
- `manuscript_text/manuscript_framing_methods_results_limitations.md`
- `audit/notebook04_final_output_audit.csv`
